# Uvicorn/JupyUvi helpers

In [ ]:
#| default_exp server

In [ ]:
# await greet('pote.server')

<!-- linkedto: _21a088dd -->
Solveit version: **0.0.85**  
dialoghelper version: **0.2.16**  
git branch: **dev-log**  
git changes: [' M ../dutil/hx_bridge.js', ' M ../explorer/htmx_bridge.ipynb', ' M 00_htmx_bridge.ipynb', '?? ../explorer/.sesskey']

Hi! I'm Vic, we're going to work together on the project **pote.server**.

In [ ]:
#| export
import asyncio, logging, subprocess
from contextlib import asynccontextmanager
from datetime import datetime
from functools import partial
from fastcore.xml import FT
from fasthtml.core import FastHTML
from fasthtml.jupyter import JupyUvi, is_port_free, HTMX
from fasthtml.xtend import Script, Link
from dialoghelper.tmux import shell_ret
from pote.dutil import set_var

In [ ]:
from fastcore.test import *
from pote.logger import logger

Please see above message with the source code for `fasthtml.jupyter`. What does it do? Explain.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠

`fasthtml.jupyter` provides utilities for running and displaying FastHTML/HTMX web apps **inside notebooks** (Jupyter, Solveit, etc.). Here's what the key pieces do:

**Server management**
- `nb_serve` / `nb_serve_async` — start a uvicorn server in a background thread (or async task) so it doesn't block the kernel. Waits until the server is actually ready before returning.
- `JupyUvi` / `JupyUviAsync` — convenient wrapper classes around the above, with `.start()` / `.stop()` lifecycle methods and automatic HTMX port configuration when not in Solveit.
- `wait_port_free` — blocks until a port is released (useful after `.stop()`).

**Display helpers**
- `show` — renders FastHTML components in the notebook, and triggers `htmx.process()` so htmx attributes on dynamically injected HTML are activated.
- `render_ft` — patches `FT._repr_html_` so FastHTML components auto-render with HTMX support when returned from a cell.
- `HTMX` — displays a running app in an **iframe** inside the notebook, either by URL or by rendering an `FT` component directly via a temp route.
- `htmx_config_port` — injects a JS snippet that rewrites relative htmx request paths to include the correct host/port, so htmx calls from the iframe hit the right server.

**WebSocket helper**
- `ws_client` — sets up a WebSocket-connected component displayed in an iframe, with a `send()` helper for pushing updates from Python to the browser.

In short: it bridges the gap between a notebook kernel and a live FastHTML web app, handling server lifecycle, port routing, and in-notebook rendering.

## helpers

In [ ]:
#| export
def kill_port(port): subprocess.run(f"lsof -ti:{port} | xargs -r kill -9", shell=True)

In [ ]:
#| export
def is_serving(srv, timeout=1):
    "Check if a JupyUvi instance is live and serving requests"
    if not (srv and srv.server and srv.server.started and not srv.server.should_exit and not is_port_free(srv.port)): return False
    try:
        import httpx
        return httpx.get(f"http://localhost:{srv.port}/", timeout=timeout).status_code < 500
    except Exception: return False

In [ ]:
#| export
def wrap_endp(response):
    "If `response` is not callable, wrap a str/FT/sequence str|FT as a async endpoint"
    if callable(response) and not isinstance(response, FT) and not hasattr(response, '__ft__'): return response
    async def endpoint(request): return response
    return endpoint

In [ ]:
#| export
def find_server(port=8000):
    "Scan IPython user_ns for a JupyUvi instance on `port`"
    import IPython
    ns = IPython.get_ipython().user_ns
    for k,v in ns.items():
        if isinstance(v, JupyUvi) and v.port == port and v.server and v.server.started and not v.server.should_exit: return v,k
    return None, None

In [ ]:
test_eq(find_server(), (None,None))

## server

In [ ]:
#| export
daisyui_hdrs = (
    Link(href='https://cdn.jsdelivr.net/npm/daisyui@5', rel='stylesheet', type='text/css'),
    Script(src='https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4'),
    Link(href='https://cdn.jsdelivr.net/npm/daisyui@5/themes.css', rel='stylesheet', type='text/css'),
)
threejs_hdrs = (
    Script('{"imports":{"three":"https://cdn.jsdelivr.net/npm/three@0.165.0/build/three.module.js","three/addons/":"https://cdn.jsdelivr.net/npm/three@0.165.0/examples/jsm/"}}', type='importmap'),
)

In [ ]:
#| export
def get_preview(app): return partial(HTMX, app=app)

def ensure_server(srv=None, port=8000, hdrs=None):
    if not srv: srv,_ = find_server(port)
    if not srv:
        app = FastHTML(hdrs=hdrs or daisyui_hdrs)
        srv = JupyUvi(app)
    else: app = srv.app
    return app, app.route, srv, get_preview(app)

def stop_server(port=8000):
    srv, sym = find_server(port)
    if is_serving(srv):
        srv.stop()
        set_var(sym, None)

In [ ]:
kill_port(8000)

In [ ]:
app, rt, srv, p = ensure_server()
test_is(srv is not None, True)

In [ ]:
app2, _, srv2, _ = ensure_server()
test_eq((app,srv), (app2,srv2))

In [ ]:
_srv, sym = find_server()
test_eq((_srv, sym), (srv, 'srv'))

In [ ]:
stop_server()

In [ ]:
test_is(globals()['srv'], None)
test_eq(find_server(), (None, None))
test_is(is_serving(srv2), False)

## test rig

In [ ]:
#| exporti
def log(msg, *args, **kwargs): print(f"[{datetime.now():%H:%M:%S}] {msg}", flush=True)

In [ ]:
log = await logger()

In [ ]:
#| exporti
class LoggerHandler(logging.Handler):
    def __init__(self, log): self.log = log; super().__init__()
    def emit(self, record): self.log(f"{(record.levelname+':'):<10}{self.format(record)}")

In [ ]:
#| export
def setup_uvi_logging(_log=None):
    _log = _log or log
    _lh = LoggerHandler(_log)
    for name in ('uvicorn', 'uvicorn.error', 'uvicorn.access'):
        lg = logging.getLogger(name)
        lg.handlers, lg.propagate = [_lh], False
        lg.setLevel(logging.DEBUG)

In [ ]:
# @asynccontextmanager
# async def local_server(app=None, host='0.0.0.0', port=8000, quiet=False, timeout=3, **uvikw):
#     "Run a Starlette app in a background thread for testing"
#     if not is_port_free(port, host): raise RuntimeError(f"Port {port} already in use")
#     app = app or Starlette()
#     srv = uvicorn.Server(uvicorn.Config(app, host=host, port=port, **uvikw))
#     thrd = threading.Thread(target=srv.run, daemon=True)
#     thrd.start()
#     deadline = (loop := asyncio.get_running_loop()).time() + timeout
#     while not srv.started:
#         if loop.time() > deadline: raise RuntimeError(f"Server failed to start on port {port} within {timeout}s")
#         await asyncio.sleep(0.01)
#     if not quiet:
#         try: print(f">>>{thrd.is_alive()=}\n{shell_ret(f'lsof -i :{port}')}")
#         except Exception as e: print(f">>>startup diag failed: {e}")
#     try: yield thrd, srv, app
#     finally:
#         srv.should_exit = True
#         thrd.join(timeout=timeout)
#         if not quiet:
#             if thrd.is_alive(): print(f"WARNING: server thread still alive after {timeout}s")
#             try: print(f"<<<{thrd.is_alive()=}\n{shell_ret(f'lsof -i :{port}')}")
#             except Exception as e: print(f"<<<teardown diag failed: {e}")

In [ ]:
# @asynccontextmanager
# async def local_server(port=8000, **kw):
#     "Yield a running JupyUvi on `port`, spinning one up if needed"
#     server = None
#     if not is_port_free(port):
#         srv = find_server(port)
#         if srv is None: raise RuntimeError(f"Port {port} in use but no JupyUvi instance found in namespace")
#     else: server = srv = JupyUvi(FastHTML(), port=port, **kw)
#     try: yield srv, srv.app, srv.app.route
#     finally:
#         if server: server.stop()

In [ ]:
#| export
@asynccontextmanager
async def local_server(app=None, port=8000, after_start=None, before_stop=None, quiet=True, timeout=5, **jupyuvi_kw):
    "Yield a running JupyUvi server that can be used for testing; spins one up if needed, then cleanly tears it down afterwards"
    server = None
    if not is_port_free(port): 
        srv,_ = find_server(port)
        if srv is None: raise RuntimeError(f"Port {port} in use but no JupyUvi instance found in namespace")
        if not is_serving(srv): raise RuntimeError(f"Port {port} in use but JupyUvi server is not usable")
    else: 
        server = srv = JupyUvi(app or FastHTML(), port=port, start=False, **jupyuvi_kw)
        await srv.start_async()
        if not quiet:
            try: print(f">>> started: {srv.server.started}\n{shell_ret(f'lsof -i :{port}')}")
            except Exception as e: print(f">>> server startup failed: {e}")
    if after_start: await after_start(srv) if asyncio.iscoroutinefunction(after_start) else after_start(srv)
    try: yield srv
    finally:
        if before_stop: await before_stop(srv) if asyncio.iscoroutinefunction(before_stop) else before_stop(srv)
        if not server: return
        srv.server.should_exit = True
        deadline = (loop := asyncio.get_running_loop()).time() + timeout
        while not is_port_free(port):
            if loop.time() > deadline:
                print(f"WARNING: port {port} still in use after {timeout}s"); break
            await asyncio.sleep(0.1)
        if not quiet:
            try: print(f"<<< after stop: port {port} {'still in use' if shell_ret(f'lsof -i :{port}') else 'free'}")
            except Exception as e: print(f"<<< teardown failed: {e}")

In [ ]:
setup_uvi_logging()
await log.show(True)

[19:30:58.559] INFO:     Started server process [2441]
[19:30:58.592] INFO:     Waiting for application startup.
[19:30:58.624] TRACE:    ASGI [1] Started scope={'type': 'lifespan', 'asgi': {'version': '3.0', 'spec_version': '2.0'}, 'state': {}}
[19:30:58.658] TRACE:    ASGI [1] Receive {'type': 'lifespan.startup'}
[19:30:58.690] TRACE:    ASGI [1] Send {'type': 'lifespan.startup.complete'}
[19:30:58.727] INFO:     Application startup complete.
[19:30:58.760] INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
[19:30:58.892] INFO:     Shutting down
[19:30:59.024] INFO:     Waiting for application shutdown.
[19:30:59.056] TRACE:    ASGI [1] Receive {'type': 'lifespan.shutdown'}
[19:30:59.091] TRACE:    ASGI [1] Send {'type': 'lifespan.shutdown.complete'}
[19:30:59.124] TRACE:    ASGI [1] Completed
[19:30:59.165] INFO:     Application shutdown complete.
[19:30:59.197] INFO:     Finished server process [2441]


In [ ]:
# Add log_config=None to prevent Uvicorn calling logging.config.dictConfig() on startup
async with local_server(log_level='trace', log_config=None, quiet=False) as test_srv:
    app = test_srv.app

>>> started: True
COMMAND  PID    USER   FD   TYPE   DEVICE SIZE/OFF NODE NAME
python  2441 solveit   69u  IPv4 71781382      0t0  TCP *:8000 (LISTEN)

<<< after stop: port 8000 free


`local_server` is an async context manager that provides a **temporary FastHTML/uvicorn server** for testing purposes, so you can access `.app`, add routes, make test requests, etc., inside the `async with` block.

It reuses an existing server if the port is already in use — finds it via `find_server()` and yields it. Raises if the port is occupied by something unknown. If the port is free, spins new server (async) using `JupyUvi` with whatever app and kwargs you pass (defaulting to a bare `FastHTML()`).

`after_start` and `before_stop` callbacks are optional hooks to let you seed state or do cleanup around the test.

After the block exits, it cleanly tears down the server (but only if *it* started it — borrowed servers are left alone), waiting up to `timeout` seconds for the port to free, with a warning if it times out.

When `quiet=False`, `local_server` optionally logs startup/shutdown diagnostics (port/process info via `lsof`).

The `log_config=None` kwarg passed in the test message prevents uvicorn from overwriting the custom `LoggerHandler` logging setup with its own `dictConfig` on startup.

In [ ]:
test_is(is_serving(test_srv), False)
test_eq(find_server(), (None, None))

# export -

In [ ]:
from pote.flakes import show_flakes
await show_flakes()

<div class="prose" markdown="1">

No warnings to report

</div>

In [ ]:
# #|hide
# #|eval: false
# from pote.dialog import dlg_export
# await dlg_export()